# 02 – Baselines + evaluation (Phase 3)

Before any real model, we score dead-simple predictions on 2024 val. Every later model has to beat them.
Uses the locked target from Phase 2 (`src/features/target.py`). Run `.venv/bin/python -m src.data.load` once first if `data/raw/` is empty.

In [ ]:
import sys
from pathlib import Path

# Let the notebook import code from src/ (the notebook lives one folder down).
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
from src.data.load import load_pbp
from src.features.qb_games import qb_games
from src.features.target import build_targets

pl.Config.set_tbl_cols(30)
from src.models.baselines import league_average, predict_constant, season_to_date, shrunk_to_date, carry_over
from src.models.evaluate import bootstrap_compare, mae

In [ ]:
# Train + val + test seasons only (2016-2025). We touch val to score; test stays untouched.
pbp = load_pbp(list(range(2016, 2026)))
games = qb_games(pbp)           # history: every QB-game, any size
targets = build_targets(games)  # what we predict and score
train = targets.filter(pl.col("split") == "train")
val = targets.filter(pl.col("split") == "val")
train.height, val.height

## Concept 1: What a baseline is

A baseline is the dumbest prediction that's still honest. It's the ruler a real model gets measured against:
if a fancy model can't beat "guess the league average for everyone", it hasn't learned anything.

The dumbest one: every QB, every week, gets the same rating — the **train** league average
(dropback-weighted). It's computed from 2016–2023 only, so it uses nothing from 2024.

In [ ]:
# The one number: train league-average EPA/dropback, weighted by dropbacks.
league_avg = (train["target"] * train["weight"]).sum() / train["weight"].sum()
print(f"league average (train, weighted): {league_avg:+.3f}")

# Worked example: how wrong that one number is, game by game, for two very different 2023 QBs.
example = (
    train.filter(pl.col("season") == 2023, pl.col("passer").is_in(["P.Mahomes", "Z.Wilson"]), pl.col("week") <= 3)
    .with_columns(pl.lit(league_avg).alias("prediction"))
    .with_columns((pl.col("target") - pl.col("prediction")).abs().alias("error"))
    .select("passer", "week", "target", "prediction", "error", "weight")
    .sort("passer", "week")
)
example

### Score it on 2024 val
The number every later model has to beat. Logged in experiments.md as the first row.

In [ ]:
# Step 1: one number from train. Step 2: give it to every val game. Step 3: score.
league_avg = league_average(train)
baseline_scores = mae(predict_constant(val, league_avg))
print(f"league-average baseline (+{league_avg:.3f} for everyone), 2024 val:")
print(f"  weighted MAE   {baseline_scores['weighted']:.3f}   <- official score")
print(f"  unweighted MAE {baseline_scores['unweighted']:.3f}")

### ▶ Play: Concept 1
Change the one number everyone gets, and see how the MAE moves. Then pick a QB and see how wrong "everyone is average" is for him.

In [ ]:
# ---- KNOBS -----------------------------------------------------------------
PREDICTION = league_avg # Try 0.0, 0.2, -0.1, or 0.074 (the 2024 median). How much does MAE move?
SHOW_QB = "L.Jackson"   # Try "J.Goff", "B.Young", "W.Levis". Who does the baseline get most wrong?
# ---------------------------------------------------------------------------

now = mae(predict_constant(val, PREDICTION))
for name, s in [(f"default (+{league_avg:.3f})", baseline_scores), (f"now ({PREDICTION:+.3f})", now)]:
    print(f"{name:20s} weighted MAE {s['weighted']:.4f}   unweighted MAE {s['unweighted']:.4f}")
print(f"change: weighted {now['weighted'] - baseline_scores['weighted']:+.4f}")

# One QB's 2024: his actual games vs the number everyone gets.
qb = predict_constant(val.filter(pl.col("passer") == SHOW_QB), PREDICTION)
if qb.height == 0:
    print(f"\nNo 2024 games for {SHOW_QB}. Names look like 'P.Mahomes'.")
else:
    s = mae(qb)
    print(f"\n{SHOW_QB} 2024: {qb.height} games, actual avg {qb['target'].mean():+.3f}, "
          f"weighted MAE with {PREDICTION:+.3f} for everyone: {s['weighted']:.3f}")

## Concept 2: Season-to-date average

Each QB's prediction for week N = his EPA/dropback over his earlier games that season (weeks < N),
like a batting average. Week 1 (no history) falls back to the train league average.
History includes 1–2 dropback games (they're real plays), and the season resets each year.

In [ ]:
std_preds = season_to_date(games, targets, fallback=league_avg)
std_scores = mae(std_preds.filter(pl.col("split") == "val"))
print("2024 val           weighted   unweighted")
print(f"league average     {baseline_scores['weighted']:.3f}      {baseline_scores['unweighted']:.3f}")
print(f"season-to-date     {std_scores['weighted']:.3f}      {std_scores['unweighted']:.3f}")

### Why is it worse? Break it down by how much history the QB has (train only)
Diagnosing on train keeps 2024 from shaping the next idea.

In [ ]:
def history_bucket(col):
    return (pl.when(col == 0).then(pl.lit("a) 0 (fallback)"))
              .when(col <= 100).then(pl.lit("b) 1-100"))
              .when(col <= 200).then(pl.lit("c) 101-200"))
              .when(col <= 300).then(pl.lit("d) 201-300"))
              .otherwise(pl.lit("e) 300+")))

tr = std_preds.filter(pl.col("split") == "train").with_columns(
    history_bucket(pl.col("prior_dropbacks").fill_null(0)).alias("dropbacks_so_far"))
rows = []
for bucket in sorted(tr["dropbacks_so_far"].unique()):
    b = tr.filter(pl.col("dropbacks_so_far") == bucket)
    rows.append({"dropbacks_so_far": bucket, "games": b.height,
                 "season_to_date": mae(b)["weighted"],
                 "league_average": mae(predict_constant(b, league_avg))["weighted"]})
pl.DataFrame(rows)

### ▶ Play: Concept 2
Don't trust a QB's own average until he has enough dropbacks; before that, use the league average.

In [ ]:
# ---- KNOBS -----------------------------------------------------------------
MIN_HISTORY = 0         # Dropbacks needed before we trust his own average. Try 100, 200, 300.
SHOW_QB = "B.Young"     # Try "L.Jackson", "J.Goff", "W.Levis". Watch his 2024 prediction move week by week.
# ---------------------------------------------------------------------------

def with_min_history(preds, n):
    return preds.with_columns(
        pl.when(pl.col("prior_dropbacks").fill_null(0) >= max(n, 1))
        .then(pl.col("prediction")).otherwise(pl.lit(league_avg)).alias("prediction"))

now_preds = with_min_history(std_preds, MIN_HISTORY)
for name, df in [("default (MIN_HISTORY=0)", std_preds), (f"now (MIN_HISTORY={MIN_HISTORY})", now_preds)]:
    s = mae(df.filter(pl.col("split") == "val"))
    print(f"{name:26s} val weighted MAE {s['weighted']:.4f}   unweighted {s['unweighted']:.4f}")
print(f"(league average for everyone: {baseline_scores['weighted']:.4f})")

qb = now_preds.filter(pl.col("split") == "val", pl.col("passer") == SHOW_QB)
if qb.height == 0:
    print(f"\nNo 2024 games for {SHOW_QB}. Names look like 'P.Mahomes'.")
else:
    print(f"\n{SHOW_QB} 2024:")
    print(qb.select("week", "prior_dropbacks", "prediction", "target", "weight"))

## Concept 3: Shrinkage

Every QB starts each season with K imaginary league-average dropbacks, then his real ones are added:
`(his EPA so far + K × league_avg) / (his dropbacks so far + K)`. K = 0 is the raw season-to-date average.

### Pick K on train only
Choosing K by its val score would let 2024 tune the model, so val would stop being an honest test.

In [ ]:
K_GRID = [0, 25, 50, 100, 150, 200, 250, 300, 400, 600, 800, 1200]
train_curve = pl.DataFrame({
    "k": K_GRID,
    "train_weighted_mae": [mae(shrunk_to_date(games, train, league_avg, k))["weighted"] for k in K_GRID],
})
BEST_K = int(train_curve.sort("train_weighted_mae")["k"][0])
print(f"best K on train: {BEST_K}")
train_curve

In [ ]:
shrunk_preds = shrunk_to_date(games, targets, league_avg, BEST_K)
shrunk_scores = mae(shrunk_preds.filter(pl.col("split") == "val"))
print("2024 val                 weighted   unweighted")
print(f"league average           {baseline_scores['weighted']:.3f}      {baseline_scores['unweighted']:.3f}")
print(f"season-to-date           {std_scores['weighted']:.3f}      {std_scores['unweighted']:.3f}")
print(f"shrinkage (K={BEST_K})       {shrunk_scores['weighted']:.3f}      {shrunk_scores['unweighted']:.3f}")

### ▶ Play: Concept 3
Turn the shrinkage dial and see the val score, and one QB's 2024 predictions, move.

In [ ]:
# ---- KNOBS -----------------------------------------------------------------
K = BEST_K              # Imaginary average dropbacks. Try 0 (raw average), 50, 1000, 100000 (≈ league average).
SHOW_QB = "B.Young"     # Try "L.Jackson", "J.Goff", "W.Levis".
# ---------------------------------------------------------------------------

now_preds = shrunk_to_date(games, targets, league_avg, K)
now = mae(now_preds.filter(pl.col("split") == "val"))
for name, s in [(f"default (K={BEST_K})", shrunk_scores), (f"now (K={K})", now)]:
    print(f"{name:20s} val weighted MAE {s['weighted']:.4f}   unweighted {s['unweighted']:.4f}")
print(f"change: weighted {now['weighted'] - shrunk_scores['weighted']:+.4f}")

qb = (now_preds.filter(pl.col("split") == "val", pl.col("passer") == SHOW_QB)
      .join(std_preds.select("game_id", "passer_id", pl.col("prediction").alias("raw_average")), on=["game_id", "passer_id"]))
if qb.height == 0:
    print(f"\nNo 2024 games for {SHOW_QB}. Names look like 'P.Mahomes'.")
else:
    print(f"\n{SHOW_QB} 2024 (prediction uses K={K}):")
    print(qb.select("week", "prior_dropbacks", "raw_average", "prediction", "target"))

## Concept 4: Carry over last season

Start each QB from his own previous season (shrunk toward the league average with the same K),
instead of from the league average. No dropbacks last season (rookies) → league average.
K is picked on train again, since the best amount of trust may change.

In [ ]:
CARRY_K_GRID = [100, 150, 200, 250, 300, 400, 600, 800]
carry_curve = pl.DataFrame({
    "k": CARRY_K_GRID,
    "train_weighted_mae": [mae(carry_over(games, train, league_avg, k))["weighted"] for k in CARRY_K_GRID],
})
CARRY_K = int(carry_curve.sort("train_weighted_mae")["k"][0])
print(f"best K on train: {CARRY_K}")
carry_curve

In [ ]:
carry_preds = carry_over(games, targets, league_avg, CARRY_K)
carry_scores = mae(carry_preds.filter(pl.col("split") == "val"))
print("2024 val                 weighted   unweighted")
print(f"league average           {baseline_scores['weighted']:.3f}      {baseline_scores['unweighted']:.3f}")
print(f"season-to-date           {std_scores['weighted']:.3f}      {std_scores['unweighted']:.3f}")
print(f"shrinkage (K={BEST_K})       {shrunk_scores['weighted']:.3f}      {shrunk_scores['unweighted']:.3f}")
print(f"carry-over (K={CARRY_K})      {carry_scores['weighted']:.3f}      {carry_scores['unweighted']:.3f}")

# Where did it help? Compare the two by week of the season.
v = (carry_preds.filter(pl.col("split") == "val")
     .join(shrunk_preds.select("game_id", "passer_id", pl.col("prediction").alias("shrunk")), on=["game_id", "passer_id"]))
rows = []
for label, cond in [("week 1", pl.col("week") == 1), ("weeks 2-4", pl.col("week").is_between(2, 4)), ("weeks 5+", pl.col("week") >= 5)]:
    b = v.filter(cond)
    rows.append({"weeks": label, "games": b.height, "carry_over": mae(b)["weighted"], "shrinkage": mae(b, "shrunk")["weighted"]})
pl.DataFrame(rows)

### ▶ Play: Concept 4
Change K for carry-over, and compare one QB's 2024 predictions with and without last season.

In [ ]:
# ---- KNOBS -----------------------------------------------------------------
K = CARRY_K             # Try 50 (trust this season fast) or 2000 (lean on last season all year).
SHOW_QB = "J.Allen"     # Try "B.Young", "D.Prescott", "J.Daniels" (a rookie: no last season).
# ---------------------------------------------------------------------------

now_preds = carry_over(games, targets, league_avg, K)
now = mae(now_preds.filter(pl.col("split") == "val"))
for name, s in [(f"default (K={CARRY_K})", carry_scores), (f"now (K={K})", now)]:
    print(f"{name:20s} val weighted MAE {s['weighted']:.4f}   unweighted {s['unweighted']:.4f}")
print(f"change: weighted {now['weighted'] - carry_scores['weighted']:+.4f}")

qb = (now_preds.filter(pl.col("split") == "val", pl.col("passer") == SHOW_QB)
      .join(shrunk_preds.select("game_id", "passer_id", pl.col("prediction").alias("shrinkage")), on=["game_id", "passer_id"]))
if qb.height == 0:
    print(f"\nNo 2024 games for {SHOW_QB}. Names look like 'P.Mahomes'.")
else:
    print(f"\n{SHOW_QB} 2024: last season {qb['last_dropbacks'][0]} dropbacks, starting point {qb['start'][0]:+.3f}")
    print(qb.select("week", "shrinkage", pl.col("prediction").alias("carry_over"), "target"))

## Concept 5: Is a small gain real? (bootstrap)

Make thousands of fake 2024 seasons by drawing QBs at random with replacement, score both models
on each, and count how often the newer one wins. Whole QBs are drawn (not single games) because one
QB's games are linked.

In [ ]:
# All four baselines side by side on 2024 val, one column each.
keys = ["game_id", "passer_id"]
val_all = (predict_constant(val, league_avg).rename({"prediction": "league_average"})
    .join(std_preds.select(*keys, pl.col("prediction").alias("season_to_date")), on=keys)
    .join(shrunk_preds.select(*keys, pl.col("prediction").alias("shrinkage")), on=keys)
    .join(carry_preds.select(*keys, pl.col("prediction").alias("carry_over")), on=keys))

rows = []
for old, new in [("league_average", "season_to_date"), ("league_average", "shrinkage"), ("shrinkage", "carry_over")]:
    r = bootstrap_compare(val_all, old, new)
    rows.append({"old": old, "new": new, "gain": r["gain"], "new_wins_%": 100 * r["win_rate"],
                 "gain_95%_low": r["low"], "gain_95%_high": r["high"]})
pl.DataFrame(rows)

### ▶ Play: Concept 5
Change what gets resampled, or how many fake seasons, and see how sure the comparison looks.

In [ ]:
# ---- KNOBS -----------------------------------------------------------------
GROUP_BY = "passer_id"  # Try "game_id" (each game its own draw) or "week" (whole weeks). Which looks most/least sure?
N_BOOT = 5000           # Try 50 or 200. Run twice with different SEED: do the answers wobble?
SEED = 0
# ---------------------------------------------------------------------------

for old, new in [("league_average", "shrinkage"), ("shrinkage", "carry_over")]:
    base = bootstrap_compare(val_all, old, new)
    now = bootstrap_compare(val_all, old, new, group_col=GROUP_BY, n_boot=N_BOOT, seed=SEED)
    print(f"{new} vs {old}:")
    print(f"  default (QBs, 5000)          {new} wins {100 * base['win_rate']:5.1f}%   95% range {base['low']:+.4f} to {base['high']:+.4f}")
    print(f"  now ({GROUP_BY}, {N_BOOT})".ljust(31) + f"{new} wins {100 * now['win_rate']:5.1f}%   95% range {now['low']:+.4f} to {now['high']:+.4f}")

## Your exploration

Add cells below.